In [0]:
# df = spark.table("dev_catalog.aqi_strm_dev.aqi")

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, when, window, max, min, avg, sum, count, lit, last, lead, lag, to_timestamp, first, row_number
from pyspark.sql.window import Window

In [0]:
# df = df.withColumn("latitude", col("latitude").try_cast("double")) \
#         .withColumn("longitude", col("longitude").try_cast("double")) \
#         .withColumn("max_value", col("max_value").try_cast("double")) \
#         .withColumn("min_value", col("min_value").try_cast("double")) \
#         .withColumn("avg_value", col("avg_value").try_cast("double")) \
#         .withColumn("last_update", to_timestamp("last_update", "dd-MM-yyyy HH:mm:ss"))

In [0]:
@dp.table

def bronze_aqi():
    env = spark.conf.get("pipelines.env", "dev")
    source_table = f"{env}_catalog.aqi_strm_{env}.aqi"
    
    return spark.readStream.table(source_table) \
        .withColumn("latitude", col("latitude").try_cast("double")) \
        .withColumn("longitude", col("longitude").try_cast("double")) \
        .withColumn("max_value", col("max_value").try_cast("double")) \
        .withColumn("min_value", col("min_value").try_cast("double")) \
        .withColumn("avg_value", col("avg_value").try_cast("double")) \
        .withColumn("last_update", to_timestamp("last_update", "yyyy-MM-dd HH:mm:ss"))

In [0]:
# agg_df = df.groupBy("country", "state", "city", "pollutant_id").agg(max("max_value").alias("max_pollutant_level"), min("min_value").alias("min_pollutant_level"), avg("avg_value").alias("avg_pollutant_level"), first("latitude").alias("latitude"), first("longitude").alias("longitude"), max("last_update").alias("last_update"))

# agg_df = agg_df.withColumn("aqi_quality_category", when(col("avg_pollutant_level") <= 50, "Good").
#                                                     when(col("avg_pollutant_level") <= 100, "Satisfactory").
#                                                     when(col("avg_pollutant_level") <= 200, "Moderate").
#                                                     when(col("avg_pollutant_level") <= 300, "Poor").
#                                                     otherwise("Severe"))
# agg_df = agg_df.withColumn("alert_level", when(col("avg_pollutant_level") >= 300, "Critical").
#                                             when(col("avg_pollutant_level") >= 200, "High").
#                                             when(col("avg_pollutant_level") >= 100, "Medium").
#                                             otherwise("Normal"))

In [0]:
silver_rules = {
    "country": "country is not null",
    "state": "state is not null",
    "city": "city is not null",
    "pollutant_id": "pollutant_id is not null",
    "avg_pollutant_level": "avg_pollutant_level is not null"
}

In [0]:
@dp.table
@dp.expect_all_or_drop(silver_rules)

def silver_aqi():
    df = spark.readStream.table("bronze_aqi")
    agg_df = df.groupBy("country", "state", "city", "pollutant_id").agg(max("max_value").alias("max_pollutant_level"), min("min_value").alias("min_pollutant_level"), avg("avg_value").alias("avg_pollutant_level"), first("latitude").alias("latitude"), first("longitude").alias("longitude"), max("last_update").alias("last_update"))

    agg_df = agg_df.withColumn("aqi_quality_category", when(col("avg_pollutant_level") <= 50, "Good").
                                                        when(col("avg_pollutant_level") <= 100, "Satisfactory").
                                                        when(col("avg_pollutant_level") <= 200, "Moderate").
                                                        when(col("avg_pollutant_level") <= 300, "Poor").
                                                        otherwise("Severe"))
    agg_df = agg_df.withColumn("alert_level", when(col("avg_pollutant_level") >= 300, "Critical").
                                                when(col("avg_pollutant_level") >= 200, "High").
                                                when(col("avg_pollutant_level") >= 100, "Medium").
                                                otherwise("Normal"))
    
    return agg_df

In [0]:
# display(agg_df)

country,state,city,pollutant_id,max_pollutant_level,min_pollutant_level,avg_pollutant_level,latitude,longitude,last_update,aqi_quality_category,alert_level
India,Andhra_Pradesh,Amaravati,NO2,24,13,16.0,16.5150833,80.5181667,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Amaravati,OZONE,77,9,41.0,16.5150833,80.5181667,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Anantapur,CO,78,25,40.0,14.675886,77.593027,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Anantapur,NH3,7,4,4.0,14.675886,77.593027,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Anantapur,OZONE,55,30,37.0,14.675886,77.593027,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Anantapur,PM10,85,53,70.0,14.675886,77.593027,19-04-2026 15:00:00,Satisfactory,Normal
India,Andhra_Pradesh,Anantapur,SO2,12,6,7.0,14.675886,77.593027,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Chittoor,NH3,8,5,6.0,13.204880,79.097889,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Chittoor,NO2,37,24,28.0,13.204880,79.097889,19-04-2026 15:00:00,Good,Normal
India,Andhra_Pradesh,Chittoor,OZONE,37,17,26.0,13.204880,79.097889,19-04-2026 15:00:00,Good,Normal


### State Wise Top Dominating Cities

In [0]:
# dom_df = agg_df
# dom_window = Window.partitionBy("State").orderBy(col("avg_pollutant_level").desc())
# dom_df = dom_df.withColumn("pollutant_dominant_rank", row_number().over(dom_window))
# dom_df = dom_df.filter(col("pollutant_dominant_rank") == 1)

In [0]:
@dp.materialized_view()

def gold_aqi_dominating_cities():
    agg_df = spark.read.table("silver_aqi")
    dom_df = agg_df
    dom_window = Window.partitionBy("State").orderBy(col("avg_pollutant_level").desc())
    dom_df = dom_df.withColumn("pollutant_dominant_rank", row_number().over(dom_window))
    dom_df = dom_df.filter(col("pollutant_dominant_rank") == 1)

    return dom_df

In [0]:
# display(dom_df)

country,state,city,pollutant_id,max_pollutant_level,min_pollutant_level,avg_pollutant_level,latitude,longitude,last_update,aqi_quality_category,alert_level,pollutant_dominant_rank
India,Andhra_Pradesh,Visakhapatnam,PM10,208,31,86.0,17.72,83.3,19-04-2026 15:00:00,Satisfactory,Normal,1
India,Arunachal_Pradesh,Naharlagun,CO,26,10,10.0,27.103358,93.679645,19-04-2026 15:00:00,Good,Normal,1
India,Assam,Byrnihat,PM2.5,320,40,100.0,26.071318,91.87488,19-04-2026 15:00:00,Satisfactory,Medium,1
India,Bihar,Manguraha,PM2.5,500,7,185.0,27.308328,84.531742,19-04-2026 15:00:00,Moderate,Medium,1
India,Chandigarh,Chandigarh,OZONE,84,3,45.5,30.735567,76.775714,19-04-2026 15:00:00,Good,Normal,1
India,Chhattisgarh,Bilaspur,PM2.5,261,100,172.0,22.08815,82.13737,19-04-2026 15:00:00,Moderate,Medium,1
India,Delhi,Delhi,PM10,500,101,251.1904761904762,28.732820,77.170633,19-04-2026 15:00:00,Poor,High,1
India,Gujarat,Bhavnagar,PM10,240,74,134.0,21.755417,72.139056,19-04-2026 15:00:00,Moderate,Medium,1
India,Haryana,Ballabgarh,PM2.5,409,44,241.0,28.3419248,77.319699,19-04-2026 15:00:00,Poor,High,1
India,Himachal Pradesh,Baddi,PM2.5,412,45,133.0,30.943887,76.801991,19-04-2026 15:00:00,Moderate,Medium,1


### Top 10 Most Polluted States

In [0]:
# mp_df = agg_df
# mp_df = mp_df.groupBy("state").agg(max("avg_pollutant_level").alias("avg_aqi")).orderBy(col("avg_aqi").desc()).limit(10)
# display(mp_df)

state,avg_aqi
Uttar_Pradesh,384.0
Madhya Pradesh,290.0
Delhi,251.1904761904762
Haryana,241.0
Meghalaya,240.0
Punjab,215.0
Odisha,203.0
Maharashtra,202.0
Rajasthan,200.5
Bihar,185.0


In [0]:
@dp.materialized_view()

def gold_aqi_most_polluted_states():
    agg_df = spark.read.table("silver_aqi")
    mp_df = agg_df
    mp_df = mp_df.groupBy("state").agg(max("avg_pollutant_level").alias("avg_aqi")).orderBy(col("avg_aqi").desc()).limit(10)
    
    return mp_df
    

### Top 10 Cities With Best AQI

In [0]:
# baqi_df = agg_df
# baqi_df = baqi_df.groupBy("state", "city").agg(avg("avg_pollutant_level").alias("avg_aqi")).orderBy(col("avg_aqi").asc()).limit(10)
# display(baqi_df)

state,city,avg_aqi
TamilNadu,Ramanathapuram,null
Bihar,Chhapra,null
Odisha,Tensa,null
Bihar,Aurangabad,null
Kerala,Thiruvananthapuram,null
Maharashtra,Malegaon,null
Jharkhand,Dhanbad,2.0
Bihar,Motihari,3.5
Assam,Nagaon,4.0
Uttar_Pradesh,Jhansi,6.0


In [0]:
@dp.materialized_view()

def gold_aqi_best_aqi():
    agg_df = spark.read.table("silver_aqi")
    baqi_df = agg_df
    baqi_df = baqi_df.groupBy("state", "city").agg(avg("avg_pollutant_level").alias("avg_aqi")).orderBy(col("avg_aqi").asc()).limit(10)
    
    return baqi_df

### Severe AQI States

In [0]:
# display(agg_df)

country,state,city,pollutant_id,max_pollutant_level,min_pollutant_level,avg_pollutant_level,latitude,longitude,last_update,aqi_quality_category,alert_level
India,Punjab,Patiala,NH3,7.0,6.0,6.0,30.349388,76.366642,2026-04-19T15:00:00.000Z,Good,Normal
India,TamilNadu,Thoothukudi,PM10,112.0,40.0,76.0,8.816428,78.099039,2026-04-19T15:00:00.000Z,Satisfactory,Normal
India,Uttar_Pradesh,Ghaziabad,CO,173.0,4.0,50.75,28.694528,77.494705,2026-04-19T15:00:00.000Z,Satisfactory,Normal
India,Uttar_Pradesh,Greater Noida,CO,134.0,8.0,21.0,28.557054,77.453663,2026-04-19T15:00:00.000Z,Good,Normal
India,Uttar_Pradesh,Raebareli,PM2.5,null,null,null,26.207781,81.245571,2026-04-19T15:00:00.000Z,Severe,Normal
India,West_Bengal,Asansol,CO,106.0,14.0,46.0,23.697936,86.944395,2026-04-19T15:00:00.000Z,Good,Normal
India,West_Bengal,Durgapur,NO2,59.0,1.0,21.5,23.508764,87.35444,2026-04-19T15:00:00.000Z,Good,Normal
India,Andhra_Pradesh,Tirupati,SO2,16.0,8.0,10.0,13.615387,79.40923,2026-04-19T15:00:00.000Z,Good,Normal
India,Bihar,Patna,CO,113.0,29.0,45.5,25.610369,85.132568,2026-04-19T15:00:00.000Z,Good,Normal
India,Assam,Silchar,CO,22.0,14.0,19.0,24.82827,92.79525,2026-04-19T15:00:00.000Z,Good,Normal


In [0]:
# sev_df = agg_df
# sev_df = sev_df.filter(col("aqi_quality_category") == "Severe")
# display(sev_df)

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-4815160942465689>, line 3
      1 sev_df = agg_df
      2 sev_df = sev_df.filter(col("aqi_quality_category") == "Severe")
----> 3 display(sev_df)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     94     self.cf_helper.display_streaming_dataframe(df, config, self.streaming_listener,
     95                                                **kwargs)
     96 else:
---> 97     self.cf_helper.display_dataframe(df, config

In [0]:
@dp.materialized_view()

def gold_aqi_severe():
    agg_df = spark.read.table("silver_aqi")
    sev_df = agg_df
    sev_df = sev_df.filter(col("aqi_quality_category") == "Severe")
    
    return sev_df